In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from plot_func import error_scatter, interval_score, coverage, performance_dist, table

folder = "interp_reg_temp"

metrics = ["total_plus", "total_minus"]
models = [
    "1", "2", "3", "4"
]

TOLERANCE=0.1

In [2]:

for metric in metrics:
    
    dfs = []
    for model in models:
                df = pd.read_csv(f"../../{folder}/ablation_results/{metric}_ablation_{model}.csv")
                df["model"] = model
                dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)
    
    fig, ax = error_scatter(df_all, TOLERANCE, tag=metric)
    save_path = f"../../{folder}/plots/error_scatter_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = interval_score(df_all, models, tag=metric)
    save_path = f"../../{folder}/plots/interval_score_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = coverage(df_all, tag=metric)
    save_path = f"../../{folder}/plots/coverage_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
/home/wiera/Documents/fullfieldvalmetrics/scripts/plots/plot_func.py:101: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_all.groupby("model")["within_pi"]
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
/home/wiera/Documents/fullfieldvalmetrics/scripts/plots/plot_func.py:101: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_all.groupby("model")["within_pi"]


In [3]:
all_results = []

for metric in metrics:

    for model in models:

        # filepath = (f"../../{folder}/ablation_results/"
        #             f"{metric}_ablation_{model_type_print}.csv"
        # )
        filepath = f"../../{folder}/ablation_summary_{model}.csv"

        df = pd.read_csv(filepath)

        # Store the model information with each result
        df["metric"] = metric
        df["model_type"] = model


        all_results.append(df)


# Combine all summary files
results = pd.concat(all_results, ignore_index=True)
print(results)

              d_type       MAE      RMSE           MAPE  mean_abs_error  \
0           sim_plus  0.200171  0.224329       1.429920        0.200171   
1          sim_minus  0.143784  0.174321       1.400029        0.143784   
2    model_form_plus  2.881731  4.302867    3161.771594        2.881731   
3   model_form_minus  2.381499  3.387298  221059.184583        2.381499   
4         total_plus  2.853333  4.160970      11.882942        2.853333   
5        total_minus  2.362520  3.343401      21.538758        2.362520   
6           sim_plus  0.247321  0.303837       1.649247        0.247321   
7          sim_minus  0.191574  0.240254       1.583902        0.191574   
8    model_form_plus  4.459800  6.460924    6419.607171        4.459800   
9   model_form_minus  3.022366  3.731128  698356.747193        3.022366   
10        total_plus  4.384924  6.292550      19.390491        4.384924   
11       total_minus  2.938336  3.655129      25.994552        2.938336   
12          sim_plus  0.2

In [4]:
best_mean_rel_error = (
    results.loc[
        results.groupby("d_type")["mean_rel_error"].idxmin()
    ]
)

print(best_mean_rel_error[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

df_to_plot = best_mean_rel_error[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_mean_rel_error.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

              d_type model_type  mean_rel_error  pi_coverage  \
21  model_form_minus          4        1.079894     0.857143   
14   model_form_plus          3        4.622183     0.714286   
13         sim_minus          3        0.013757     0.857143   
0           sim_plus          1        0.014299     1.000000   
17       total_minus          3        0.076441     0.857143   
16        total_plus          3        0.088187     0.714286   

    mean_interval_score  
21            59.391362  
14            66.638833  
13             1.195595  
0              1.291203  
17            54.300085  
16            59.039106  


In [5]:
best_interval_score = (
    results.loc[
        results.groupby("d_type")["mean_interval_score"].idxmin()
    ]
)

print(best_interval_score[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

df_to_plot = best_interval_score[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_interval_score.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

              d_type model_type  mean_rel_error  pi_coverage  \
15  model_form_minus          3        1.855485     0.857143   
2    model_form_plus          1       31.617716     0.857143   
1          sim_minus          1        0.014000     1.000000   
12          sim_plus          3        0.014608     1.000000   
17       total_minus          3        0.076441     0.857143   
16        total_plus          3        0.088187     0.714286   

    mean_interval_score  
15            57.376084  
2             63.499753  
1              1.065298  
12             0.932341  
17            54.300085  
16            59.039106  


In [6]:
target_coverage = 0.95

results["coverage_distance"] = (
    results["pi_coverage"] - target_coverage
).abs()

best_coverage = (
    results.loc[
        results.groupby("d_type")["coverage_distance"].idxmin()
    ]
)

print(best_coverage[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

df_to_plot = best_coverage[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_coverage.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)


             d_type model_type  mean_rel_error  pi_coverage  \
3  model_form_minus          1     2210.591846     0.857143   
2   model_form_plus          1       31.617716     0.857143   
1         sim_minus          1        0.014000     1.000000   
0          sim_plus          1        0.014299     1.000000   
5       total_minus          1        0.215388     0.857143   
4        total_plus          1        0.118829     0.857143   

   mean_interval_score  
3            63.399568  
2            63.499753  
1             1.065298  
0             1.291203  
5            59.727584  
4            59.803150  


In [7]:
results["rel_error_rank"] = (
    results.groupby("d_type")["mean_rel_error"]
    .rank(method="min", ascending=True)
)

results["interval_score_rank"] = (
    results.groupby("d_type")["mean_interval_score"]
    .rank(method="min", ascending=True)
)

results["coverage_rank"] = (
    results.groupby("d_type")["coverage_distance"]
    .rank(method="min", ascending=True)
)

results["overall_rank"] = (
    results["rel_error_rank"]
    + results["interval_score_rank"]
    + results["coverage_rank"]
)

In [8]:
best_overall = (
    results.loc[
        results.groupby("d_type")["overall_rank"].idxmin()
    ]
)

print(best_overall[
    [
        "d_type",
        "model_type",
        "mean_rel_error",
        "pi_coverage",
        "mean_interval_score",
        "overall_rank",
    ]
])


df_to_plot = best_overall[
    [
        "d_type",
        "model_type",
        "mean_rel_error",
        "pi_coverage",
        "mean_interval_score",
        "overall_rank",
    ]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_overall.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

              d_type model_type  mean_rel_error  pi_coverage  \
15  model_form_minus          3        1.855485     0.857143   
2    model_form_plus          1       31.617716     0.857143   
1          sim_minus          1        0.014000     1.000000   
0           sim_plus          1        0.014299     1.000000   
17       total_minus          3        0.076441     0.857143   
16        total_plus          3        0.088187     0.714286   

    mean_interval_score  overall_rank  
15            57.376084           5.0  
2             63.499753           7.0  
1              1.065298           5.0  
0              1.291203           5.0  
17            54.300085           3.0  
16            59.039106           5.0  


In [9]:
metrics_to_plot = {
    "mean_rel_error",
    "pi_coverage",
    "mean_interval_score",
}

for model_type in results["model_type"].unique():

    kernel_results = results[
        results["model_type"] == model_type
    ].copy()

    fig, axes = performance_dist(kernel_results, metrics_to_plot, model_type)
    save_path = f"../../{folder}/plots/perform_dist_{model_type}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)